In [30]:
import os
import warnings
import pandas as pd
import numpy as np

# ML Models & Metrics
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

data_path = os.path.join('data', 'all_months_features.csv')
df_raw = pd.read_csv(data_path)

In [31]:
# Chronological sorting & Target shifting (t + 1)
df = df_raw.copy()
df['month_idx'] = df.groupby('Product_Name').cumcount() + 1
df = df.sort_values(by=['Product_Name', 'month_idx']).reset_index(drop=True)

targets_base = ['Min_Price', 'Avg_Price', 'Max_Price']
targets_next = ['Min_Price_next', 'Avg_Price_next', 'Max_Price_next']

for base_col, next_col in zip(targets_base, targets_next):
    df[next_col] = df.groupby('Product_Name')[base_col].shift(-1)

df_clean = df.dropna(subset=targets_next).copy().reset_index(drop=True)

In [32]:
#  Separate Features and Targets
non_feature_cols = [
    'Product_Name', 'Category', 'Unit', 'unit_canonical', 'month_name',
    'bs_year', 'bs_month'
] + targets_base + targets_next

feature_cols = [col for col in df_clean.columns if col not in non_feature_cols]

X = df_clean[feature_cols].copy().apply(pd.to_numeric, errors='coerce').fillna(0)
y = df_clean[targets_next].copy().fillna(0)

In [33]:
# Temporal Train/Test Split (Months 1-8 Train, Months 9-10 Test)
train_mask = (df_clean['month_idx'] <= 8).values
test_mask = (df_clean['month_idx'] >= 9).values

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Find index for Avg_Price_next
avg_idx = targets_next.index('Avg_Price_next')

In [34]:
# RANDOM FOREST: Average Price Prediction
print(" RANDOM FOREST (Avg_Price_next) ")
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model = MultiOutputRegressor(rf_base)
rf_model.fit(X_train, y_train)

y_pred_rf_test = rf_model.predict(X_test)
actual_rf = y_test.iloc[:, avg_idx].values
pred_rf = y_pred_rf_test[:, avg_idx]

rf_mae = mean_absolute_error(actual_rf, pred_rf)
rf_rmse = np.sqrt(mean_squared_error(actual_rf, pred_rf))
rf_r2 = r2_score(actual_rf, pred_rf)
print(f"[RF Test] Avg_Price_next -> MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}, R²: {rf_r2:.3f}")

 RANDOM FOREST (Avg_Price_next) 
[RF Test] Avg_Price_next -> MAE: 44.29, RMSE: 82.62, R²: 0.561


In [35]:
# LIGHTGBM: Average Price Prediction

print("\nLIGHTGBM (Avg_Price_next)")
lgb_base = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.01, num_leaves=31, random_state=42, n_jobs=-1, verbose=-1)
lgb_model = MultiOutputRegressor(lgb_base)
lgb_model.fit(X_train, y_train)

y_pred_lgb_test = lgb_model.predict(X_test)
# Constraint enforcement (Min <= Avg <= Max)
y_pred_lgb_test[:, 0] = np.minimum(y_pred_lgb_test[:, 0], y_pred_lgb_test[:, 1])
y_pred_lgb_test[:, 2] = np.maximum(y_pred_lgb_test[:, 2], y_pred_lgb_test[:, 1])

actual_lgb = y_test.iloc[:, avg_idx].values
pred_lgb = y_pred_lgb_test[:, avg_idx]

lgb_mae = mean_absolute_error(actual_lgb, pred_lgb)
lgb_rmse = np.sqrt(mean_squared_error(actual_lgb, pred_lgb))
lgb_r2 = r2_score(actual_lgb, pred_lgb)
print(f"[LightGBM Test] Avg_Price_next -> MAE: {lgb_mae:.2f}, RMSE: {lgb_rmse:.2f}, R²: {lgb_r2:.3f}")


LIGHTGBM (Avg_Price_next)
[LightGBM Test] Avg_Price_next -> MAE: 37.17, RMSE: 74.76, R²: 0.640


In [36]:
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Suppress optuna logging clutter if desired
optuna.logging.set_verbosity(optuna.logging.WARN)

# Find index for Avg_Price_next
avg_idx = targets_next.index('Avg_Price_next')

def objective(trial):
    # Define hyperparameter search space
    params = {
        'objective': 'regression_l1',
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    # Wrap base LightGBM regressor in MultiOutputRegressor to handle multi-targets
    lgb_base = lgb.LGBMRegressor(**params)
    multi_lgb = MultiOutputRegressor(lgb_base)
    
    # Train model on temporal training set
    multi_lgb.fit(X_train, y_train)
    
    # Predict on test/validation set
    y_pred = multi_lgb.predict(X_test)
    
    # Extract predictions specifically for Avg_Price_next
    pred_avg = y_pred[:, avg_idx]
    actual_avg = y_test.iloc[:, avg_idx].values
    
    # Calculate MAE for the target of interest
    mae = mean_absolute_error(actual_avg, pred_avg)
    return mae

# Run the Optuna study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n--- BEST OPTUNA PARAMETERS ---")
print(study.best_params)
print(f"Best Test MAE (Avg_Price_next): {study.best_value:.4f}")

# Train final model using the best parameters found
best_params = study.best_params
best_params.update({'objective': 'regression_l1', 'random_state': 42, 'n_jobs': -1, 'verbose': -1})

final_lgb_base = lgb.LGBMRegressor(**best_params)
final_model = MultiOutputRegressor(final_lgb_base)
final_model.fit(X_train, y_train)

# Final Evaluation
y_pred_final = final_model.predict(X_test)
pred_final_avg = y_pred_final[:, avg_idx]
actual_final_avg = y_test.iloc[:, avg_idx].values

final_mae = mean_absolute_error(actual_final_avg, pred_final_avg)
final_rmse = np.sqrt(mean_squared_error(actual_final_avg, pred_final_avg))
final_r2 = r2_score(actual_final_avg, pred_final_avg)

print(f"\n[Optimized LightGBM Test] Avg_Price_next -> MAE: {final_mae:.2f}, RMSE: {final_rmse:.2f}, R²: {final_r2:.3f}")

Best trial: 23. Best value: 31.4745: 100%|██████████| 30/30 [00:32<00:00,  1.10s/it]



--- BEST OPTUNA PARAMETERS ---
{'num_leaves': 47, 'learning_rate': 0.07414244958453477, 'subsample': 0.8111264792128177, 'colsample_bytree': 0.9080911818564337, 'reg_alpha': 0.05992298817290395, 'reg_lambda': 0.9006353739771458, 'n_estimators': 494}
Best Test MAE (Avg_Price_next): 31.4745

[Optimized LightGBM Test] Avg_Price_next -> MAE: 31.47, RMSE: 68.15, R²: 0.701


In [37]:
# Diagnostic code to check residuals and analyze errors by category/product
import pandas as pd
import numpy as np

# Create a diagnostic dataframe for the test set
diagnostic_df = pd.DataFrame({
    'Product_Name': df_clean.loc[test_mask, 'Product_Name'].values,
    'Category': df_clean.loc[test_mask, 'Category'].values,
    'Actual_Avg': actual_final_avg,
    'Predicted_Avg': pred_final_avg
})

# Calculate error metrics
diagnostic_df['Error'] = diagnostic_df['Actual_Avg'] - diagnostic_df['Predicted_Avg']
diagnostic_df['Absolute_Error'] = np.abs(diagnostic_df['Error'])
diagnostic_df['Squared_Error'] = diagnostic_df['Error'] ** 2

# 1. View top 10 worst predictions (highest absolute error)
print("--- Top 10 Worst Predictions ---")
worst_predictions = diagnostic_df.sort_values(by='Absolute_Error', ascending=False).head(10)
print(worst_predictions[['Product_Name', 'Category', 'Actual_Avg', 'Predicted_Avg', 'Absolute_Error']])

# 2. Check average error grouped by Category to find problematic sectors
print("\n--- Error Breakdown by Category ---")
category_errors = diagnostic_df.groupby('Category').agg(
    Mean_Absolute_Error=('Absolute_Error', 'mean'),
    Root_Mean_Squared_Error=('Squared_Error', lambda x: np.sqrt(np.mean(x))),
    Count=('Product_Name', 'count')
).sort_values(by='Mean_Absolute_Error', ascending=False)
print(category_errors)

--- Top 10 Worst Predictions ---
        Product_Name   Category  Actual_Avg  Predicted_Avg  Absolute_Error
2            Avocado      fruit      757.61     358.141222      399.468778
28             Mango      fruit      198.26     330.860819      132.600819
26             Lemon      fruit      363.04     235.399659      127.640341
29          Mushroom  vegetable      219.35     125.717506       93.632494
33            Orange      fruit      209.57     150.260737       59.309263
5       Bitter_Gourd  vegetable       65.00     124.185802       59.185802
36        Pine_Apple      fruit      188.91     132.795943       56.114057
24  Khasa_Garlic_Dry  vegetable      233.70     275.910352       42.210352
42      Smooth_Gourd  vegetable       78.26     117.922031       39.662031
19     Dragon_Fruits      fruit      325.00     362.065685       37.065685

--- Error Breakdown by Category ---
           Mean_Absolute_Error  Root_Mean_Squared_Error  Count
Category                                  